<a href="https://colab.research.google.com/github/hyunkyung31/coronary-ai-ml-dl/blob/main/hyunkyung/03_RF_mock_%EA%B2%80%EC%A6%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# 1단계: 최종 저장 폴더와 mock 생성에 필요한 파일 확인

In [ ]:
from pathlib import Path

FINAL_RF_DIR = Path(
    "/content/drive/MyDrive/clinical_rf_validation/"
    "rf_nested_20260828_030434/"
    "final_rf_sigmoid_20260828_072752"
)

assert FINAL_RF_DIR.exists(), (
    f"최종 모델 폴더를 찾을 수 없습니다:\n{FINAL_RF_DIR}"
)

saved_files = sorted(
    path for path in FINAL_RF_DIR.rglob("*")
    if path.is_file()
)

print("✅ 최종 모델 폴더 확인")
print(f"저장 위치: {FINAL_RF_DIR}")
print(f"파일 개수: {len(saved_files)}\n")

for path in saved_files:
    relative_path = path.relative_to(FINAL_RF_DIR)
    file_size_kb = path.stat().st_size / 1024
    print(f"- {relative_path} ({file_size_kb:,.1f} KB)")

# 2단계: 모델 무결성·threshold·입력 템플릿 확인

In [ ]:
import json
import hashlib
import pandas as pd

# 파일 경로
MODEL_PATH = FINAL_RF_DIR / "final_rf_sigmoid_model.joblib"
METADATA_PATH = FINAL_RF_DIR / "final_model_metadata.json"
THRESHOLD_PATH = FINAL_RF_DIR / "calibration_thresholds.json"
SCHEMA_PATH = FINAL_RF_DIR / "input_schema.csv"
TEMPLATE_PATH = FINAL_RF_DIR / "new_patient_template.csv"

# 1. 저장 모델 SHA-256 재확인
with open(MODEL_PATH, "rb") as file:
    model_sha256 = hashlib.sha256(file.read()).hexdigest()

expected_sha256 = (
    "43fdf7f22dd7aed1d9d2338325b67018"
    "b9394d9a756ca75e88f141a595d451ac"
)

assert model_sha256 == expected_sha256, (
    "저장된 모델의 SHA-256이 기존 기록과 다릅니다."
)

# 2. 저장 설정 불러오기
with open(METADATA_PATH, "r", encoding="utf-8") as file:
    final_metadata = json.load(file)

with open(THRESHOLD_PATH, "r", encoding="utf-8") as file:
    calibration_thresholds = json.load(file)

input_schema = pd.read_csv(SCHEMA_PATH)
new_patient_template = pd.read_csv(TEMPLATE_PATH)

print("✅ 모델 SHA-256 일치")
print("SHA-256:", model_sha256)

print("\n✅ 저장 threshold")
print(calibration_thresholds)

print("\n✅ Metadata 주요 항목")
print("저장된 항목:", list(final_metadata.keys()))

print("\n✅ 입력 스키마")
print("shape:", input_schema.shape)
display(input_schema)

print("\n✅ 신규 환자 템플릿")
print("shape:", new_patient_template.shape)
display(new_patient_template)

# 3단계: 정상 mock 환자 3명 생성 및 스키마 검사